### Реализуйте класс для реализации алгоритма градиентного спуска

learn(self)

метод возвращает итоговую среднеквадратическую ошибку.
метод итеративно вычисляет среднеквадратическую ошибку и вектор-градиент. номер итерации и MSE записываются в словарь *iteration_loss_dict*. критерий останова срабатывает тогда, когда абсолютное значение разницы двух последних MSE меньше *self.threshold*.

In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
import numpy as np
import pandas as pd

class GradientDescentMse:
   
# - признаки, # - реал. значение , # скорость , # порог (если меньше то стоп)
    def __init__(self, samples: pd.DataFrame, targets: pd.DataFrame,
                 learning_rate: float = 1e-3, threshold=1e-6):
      
        self.samples = samples.copy() # копируем таблицу, что бы не изменить исходные данные 
        self.targets = targets.values.flatten() # делаем одномерным массивом (вектором)
        self.learning_rate = learning_rate # дублируем то же значение
        self.threshold = threshold # порог оставляем
        self.iteration_loss_dict = {} # словарь MSE, после каждой итерации

        
        self.add_constant_feature() # свободный член (b)

    def add_constant_feature(self):
      
        self.samples.insert(0, 'intercept', 1) #столбец из 1, для bias (постоянного смещения)
        self.beta = np.ones(self.samples.shape[1]) # вектор b


    def calculate_mse_loss(self) -> float:
        
        predictions = self.samples.values @ self.beta #мат признаки на вектор коэф  
        residuals = predictions - self.targets #разница между реальным значением и предсказанием 
        mse = np.mean(residuals ** 2) # Квадрат нужен чтобы ошибки были положительные и большие влияли сильнее.
        return mse

    def calculate_gradient(self) -> np.ndarray:
       # считаем направление градиента 
        predictions = self.samples.values @ self.beta
        residuals = predictions - self.targets
        gradient = (2 / len(self.targets)) * (self.samples.values.T @ residuals) # как каждый влияет на ошибку (меняем местами столбцы и строки)
        return gradient

    def iteration(self):
       
        gradient = self.calculate_gradient()
        self.beta = self.beta - self.learning_rate * gradient #шагаем вниз для уменьшения ошибки

    def learn(self):
        
        iteration = 0
        previous_mse = self.calculate_mse_loss() # стартовая ошибки
        self.iteration_loss_dict[iteration] = previous_mse

        while True:
            self.iteration()
            iteration += 1
            current_mse = self.calculate_mse_loss()
            self.iteration_loss_dict[iteration] = current_mse

            if abs(previous_mse - current_mse) < self.threshold:
                break

            previous_mse = current_mse

        return current_mse
    

np.random.seed(42)  

X = pd.DataFrame({
    'feature1': np.random.rand(100),
    'feature2': np.random.rand(100),
    'feature3': np.random.rand(100)
})


true_beta = np.array([3, 2, -1, 0.5])  
noise = np.random.normal(0, 0.1, size=100) #небольшая погрешность 
y = true_beta[0] + true_beta[1]*X['feature1'] + true_beta[2]*X['feature2'] + true_beta[3]*X['feature3'] + noise
y = pd.DataFrame(y, columns=['target'])

gd = GradientDescentMse(samples=X, targets=y, learning_rate=0.1, threshold=1e-8)

final_mse = gd.learn()

print("Итоговое MSE:", final_mse)
print("Оцененные коэффициенты beta:", gd.beta)
print("Словарь с потерями по итерациям (первые 5):", dict(list(gd.iteration_loss_dict.items())[:5]))



Итоговое MSE: 0.00928350489017101
Оцененные коэффициенты beta: [ 2.95745706  2.04288343 -0.97736275  0.53475444]
Словарь с потерями по итерациям (первые 5): {0: np.float64(1.9433283890758206), 1: np.float64(1.1102023255132176), 2: np.float64(0.7537113604248752), 3: np.float64(0.5965714677278751), 4: np.float64(0.5229591985290694)}
